# 第12章　利率互换

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch12_swaps.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch12_swaps.ipynb)

复现例15.1（平价互换利率）、例15.2-15.3（估值与DV01）、例15.4（bootstrap互换曲线）、图12-1，并与 QuantLib VanillaSwap 对拍。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import swap as sw
from fi import plotting
plotting.use_chinese_style()


## 例15.1-15.3　平价互换利率、估值与 DV01（3% 平坦曲线）


In [ ]:
taus = [1, 1, 1, 1, 1]
dfs = [(1.03) ** -t for t in range(1, 6)]
print(f'平价互换利率 = {sw.par_swap_rate(dfs, taus)*100:.4f}%')
v = sw.swap_value(0.025, dfs, taus, notional=1e8, payer=True)
print(f'payer互换价值(付2.5%) = {v:,.0f} 元')
print(f'receiver互换价值      = {sw.swap_value(0.025, dfs, taus, 1e8, payer=False):,.0f} 元')
print(f'互换 DV01 = {sw.swap_dv01(dfs, taus, 1e8):,.2f} 元/bp')


## 例15.4　由互换报价 bootstrap 互换曲线


In [ ]:
par = [0.024, 0.026, 0.028, 0.030, 0.031]
z, df = sw.bootstrap_swap_curve(par)
print(f"{'期限':>4}{'par互换':>9}{'即期':>10}{'折现因子':>11}")
for n, (p, zi, d) in enumerate(zip(par, z, df), start=1):
    print(f'{n:>4}{p*100:>8.1f}%{zi*100:>9.4f}%{d:>11.6f}')
print('验证: 用DF重算5y平价互换 =', round(sw.par_swap_rate(df, [1]*5)*100, 4), '%')


## 图12-1　payer 互换盯市价值随市场利率变动（编程实验 8）


In [ ]:
rates = np.linspace(0.01, 0.05, 41)
vals = [sw.swap_value(0.025, [(1+z)**-t for t in range(1,6)], taus, 1e8, payer=True)/1e4 for z in rates]
fig, ax = plotting.new_axes()
ax.plot(rates*100, vals, label='payer 互换（付固定 2.5%）')
ax.axhline(0, ls=':', color='gray'); ax.axvline(2.5, ls=':', color='gray')
ax.set_xlabel('市场利率 (%)'); ax.set_ylabel('互换盯市价值（万元）')
ax.set_title('图12-1　payer 互换价值随市场利率变动'); ax.legend()
fig.tight_layout()


## 15.6　QuantLib VanillaSwap 对拍（编程实验 9）


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026); ql.Settings.instance().evaluationDate = today
dc, cal = ql.Actual365Fixed(), ql.NullCalendar()
disc = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.03, dc))
idx = ql.IborIndex('Idx', ql.Period(1, ql.Years), 0, ql.CNYCurrency(), cal,
                   ql.Unadjusted, False, dc, disc)
sched = ql.Schedule(today, today + ql.Period(5, ql.Years), ql.Period(1, ql.Years), cal,
                    ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Forward, False)
qswap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1e8, sched, 0.025, dc, sched, idx, 0.0, dc)
qswap.setPricingEngine(ql.DiscountingSwapEngine(disc))
print(f'fi.swap   payer NPV = {v:,.0f} 元   平价利率 = 3.0000%')
print(f'QuantLib  payer NPV = {qswap.NPV():,.0f} 元   平价利率 = {qswap.fairRate()*100:.4f}%')
print('（差异来自计息惯例/真实日历）')


---

> 小结：利率互换交换固定与浮动利息、不换本金；平价互换利率=(1−DF(Tn))/年金；
> payer 利率升则获利，DV01=年金×名义×1bp；互换曲线 bootstrap 与债券 par 曲线同形。
